In [1]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
from PIL import Image

# 1. 适配 Kaggle 环境的输入路径 
# 提示：具体层级可在右侧 Data 面板展开数据集确认，直接点击对应文件夹旁边的复制图标获取精准路径
BASE_PATH = '/kaggle/input/plant-species-dataset/2026task1/dataset-for-task2/dataset-for-task2'
TRAIN_DIR = os.path.join(BASE_PATH, 'train')
TEST_DIR = os.path.join(BASE_PATH, 'test')

# 必须将生成的文件保存在 /kaggle/working/ 下
SUBMISSION_FILE = '/kaggle/working/submission-for-task2.csv'

# （以下代码包含数据增强、模型定义、训练函数和预测函数，与此前提供的逻辑完全一致）
TRAIN_DIR = '2026task1/dataset-for-task2/dataset-for-task2/train'
TEST_DIR = '2026task1/dataset-for-task2/dataset-for-task2/test'
SUBMISSION_FILE = 'submission-for-task2.csv'

# 2. 图像增强与预处理流水线
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. 加载训练集数据
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
class_names = train_dataset.classes  # 自动读取5个子目录名称作为类别

# 4. 构建迁移学习模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names)) # 修改最后的全连接层输出为5类
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 自定义测试集数据加载器
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        # 仅读取 png 文件
        self.image_files = [f for f in os.listdir(test_dir) if f.lower().endswith('.png')]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        # 强制转换为RGB，防止含有透明通道（Alpha）的 PNG 报错
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, img_name

def train_model(epochs=10):
    print(f"开始训练，使用设备: {device}...")
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - 平均损失: {running_loss/len(train_loader):.4f}")

def generate_submission():
    print("开始对测试集进行预测...")
    test_dataset = TestDataset(TEST_DIR, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    model.eval()
    predictions = []

    with torch.no_grad():
        for inputs, file_names in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(preds)):
                predictions.append({
                    'file': file_names[i],
                    'species': class_names[preds[i].item()]
                })

    # 生成规范的提交文件
    df = pd.DataFrame(predictions)
    df.to_csv(SUBMISSION_FILE, index=False)
    print(f"预测完成！结果已保存至同级目录下的 {SUBMISSION_FILE}")

if __name__ == '__main__':
    train_model(epochs=10) # 建议根据实际收敛情况将 Epochs 提高至 20-30
    generate_submission()

FileNotFoundError: [Errno 2] No such file or directory: '2026task1/dataset-for-task2/dataset-for-task2/train'